# M3L2 E00 - PromptTemplate: del string al componente (Resolution)

## Que vamos a ver

Este notebook muestra el primer paso para pasar de un script tactico a un pipeline orquestado:
reemplazar la concatenacion manual de strings por un `ChatPromptTemplate`.

## Este notebook NO necesita API key

Solo construimos y formateamos prompts. No llamamos al modelo todavia.


## Mapa de conceptos

| Concepto de la lecture | Pregunta guia | En este notebook |
|---|---|---|
| PromptTemplate | Como estructuro un prompt sin hardcodear texto? | `ChatPromptTemplate.from_messages()` |
| Variables explicitas | Cuales son los inputs del prompt? | `{context}` y `{question}` |
| Reutilizacion | Puedo usar el mismo prompt en varios lugares? | Si, el template es un objeto |
| Modularidad | Puedo cambiar el prompt sin tocar el modelo? | Si, son componentes separados |


## Bloque 1 - El problema: string concatenado


In [ ]:
context = "La politica de vacaciones es de 15 dias por ano."
question = "Cuantos dias de vacaciones tengo?"

prompt_v1 = f"Contexto: {context}\nPregunta: {question}"
print("Version 1 (f-string simple):")
print(prompt_v1)
print()

system_msg = "Eres un asistente de RRHH. Responde solo con el contexto dado."
prompt_v2 = system_msg + "\n\n" + "Contexto: " + context + "\n\nPregunta: " + question
print("Version 2 (concatenacion):")
print(prompt_v2)


## Bloque 2 - La solucion: ChatPromptTemplate


In [ ]:
from langchain_core.prompts import ChatPromptTemplate


In [ ]:
def build_rag_prompt() -> ChatPromptTemplate:
    """
    Construye y devuelve un ChatPromptTemplate para responder preguntas con contexto.
    """
    return ChatPromptTemplate.from_messages([
        (
            "system",
            "Eres un asistente de RRHH. "
            "Responde usando solo el contexto proporcionado. "
            "Si la respuesta no esta en el contexto, indica que no tienes informacion suficiente."
        ),
        (
            "human",
            "Contexto:\n{context}\n\nPregunta:\n{question}"
        )
    ])


prompt = build_rag_prompt()
print(f"Tipo del prompt: {type(prompt)}")
print(f"Variables del prompt: {prompt.input_variables}")


In [ ]:
context_test = "La politica de vacaciones es de 15 dias por ano."
question_test = "Cuantos dias de vacaciones tengo?"

messages = prompt.format_messages(context=context_test, question=question_test)

for msg in messages:
    print(f"[{msg.type.upper()}] {msg.content}")
    print()


In [ ]:
context_rrhh = "Los empleados pueden tomar hasta 3 dias de licencia por enfermedad sin certificado."
question_rrhh = "Necesito tomar un dia por enfermedad. Que necesito presentar?"

messages_rrhh = prompt.format_messages(context=context_rrhh, question=question_rrhh)
for msg in messages_rrhh:
    print(f"[{msg.type.upper()}] {msg.content}")
    print()

print("El mismo template, distintos datos. El formato es siempre consistente.")


## Bloque 3 - Inspeccion del template


In [ ]:
print(f"Variables requeridas: {prompt.input_variables}")
print(f"Mensajes del template: {len(prompt.messages)}")
for i, msg in enumerate(prompt.messages):
    print(f"  Mensaje {i}: tipo={msg.__class__.__name__}")
    print(f"            contenido={str(msg.prompt)[:60]}...")
print()
print("Con un f-string no puedo saber cuales son las variables sin leer el codigo.")
print("Con ChatPromptTemplate puedo inspeccionar el objeto programaticamente.")


## Checks


In [ ]:
def run_checks():
    assert prompt is not None
    assert isinstance(prompt, ChatPromptTemplate)
    assert "context" in prompt.input_variables
    assert "question" in prompt.input_variables
    assert len(prompt.messages) >= 2
    msgs = prompt.format_messages(context="Contexto de prueba.", question="Pregunta de prueba?")
    assert len(msgs) >= 2
    assert "Contexto de prueba." in msgs[-1].content
    assert "Pregunta de prueba?" in msgs[-1].content
    print("M3L2 E00 Resolution checks passed")


run_checks()


## Cierre

| Sin LangChain | Con LangChain (ChatPromptTemplate) |
|---|---|
| Prompt hardcodeado en el flujo. | Prompt como objeto separado. |
| Variables implicitas en el f-string. | Variables explicitas e inspeccionables. |
| Cambio = buscar en todo el codigo. | Cambio = modificar el objeto template. |
| No se puede probar el prompt solo. | Se puede formatear y verificar antes del modelo. |

### Proximo paso: E01

En E01 conectamos este `ChatPromptTemplate` con `ChatOpenAI` usando LCEL.
